In [6]:
import csv
import psycopg2

def map_contact_source(value):
    if not value or value.strip() == '':
        return 'Other'
    return value.strip()

def map_contact_type(value):
    v = (value or '').strip().lower()
    if not v:
        return 'Other Social Media'
    if v == 'teléfono' or v == 'telefono':
        return 'Phone'
    if v == 'correo':
        return 'Email'
    return 'Other'

def map_contact_label(value):
    v = (value or '').strip().lower()
    if not v:
        return 'Personal'
    if v == 'móvil' or v == 'movil':
        return 'Mobile'
    if v == 'fijo':
        return 'Landline'
    return 'Other'

def parse_bool(value):
    return str(value).strip().lower() in ['1', 'true', 'sí', 'si', 'x']

def main():
    conn = psycopg2.connect(
        dbname='collection_db',
        user='cobranza',
        password='cobranza2025',
        host='69.48.206.219',
        port='5432'
    )
    cur = conn.cursor()

    with open('../data/debtor_contacts.csv', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            # Saltar si no hay valor de contacto
            if not row['Valor_MedioContacto'] or row['Valor_MedioContacto'].strip() == '':
                continue
            cur.execute("""
                INSERT INTO collection.debtor_contacts (
                    debtor_id, contact_source, source_notes, contact_type,
                    contact_value, contact_label, is_primary, has_contact
                ) VALUES (
                    %(debtor_id)s, %(contact_source)s, %(source_notes)s, %(contact_type)s,
                    %(contact_value)s, %(contact_label)s, %(is_primary)s, %(has_contact)s
                )
                ON CONFLICT (debtor_id, contact_type, contact_value) DO NOTHING
            """, {
                'debtor_id': int(row['ID_Deudor']),
                'contact_source': map_contact_source(row.get('Fuente_Contacto')),
                'source_notes': row.get('Nota_Fuente', None),
                'contact_type': map_contact_type(row.get('Tipo_MedioContacto')),
                'contact_value': row['Valor_MedioContacto'].strip(),
                'contact_label': map_contact_label(row.get('Etiqueta_MedioContacto')),
                'is_primary': parse_bool(row.get('Es_Principal')),
                'has_contact': False
            })
    conn.commit()
    cur.close()
    conn.close()
    print("Contactos insertados correctamente.")

if __name__ == '__main__':
    main()

Contactos insertados correctamente.
